# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DipeshGhimire33/Flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [33]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "search_volume",
    "avg_position",
    "ctr"
]

X = df[feature_cols].copy()

# Add missingness flags before filling
for col in feature_cols:
    X[f"{col}_missing"] = X[col].isna().astype(int)

# Fill numeric missing values with median
for col in feature_cols:
    X[col] = X[col].fillna(X[col].median())

# avg_position = 0 means no data, so treat it as missing
X["avg_position_missing"] = (
    df["avg_position"].eq(0).astype(int)
)

X["avg_position"] = X["avg_position"].replace(0, np.nan)
X["avg_position"] = X["avg_position"].fillna(X["avg_position"].median())

In [34]:
print("Feature shape:", X.shape) 
display(X.head())

Feature shape: (30000, 20)


,days_since_last_update,impressions_90d,clicks_90d,impressions_last_30d,clicks_last_30d,impressions_prev_30d,clicks_prev_30d,search_volume,avg_position,ctr,days_since_last_update_missing,impressions_90d_missing,clicks_90d_missing,impressions_last_30d_missing,clicks_last_30d_missing,impressions_prev_30d_missing,clicks_prev_30d_missing,search_volume_missing,avg_position_missing,ctr_missing
0,20,3803,29,578,2,987,13,10.0,10.6,0.76,0,0,0,0,0,0,0,0,0,0
1,25,15320,7,2501,2,5915,1,90.0,20.3,0.05,0,0,0,0,0,0,0,0,0,0
2,20,12581,11,2382,1,6089,3,0.0,36.5,0.09,0,0,0,0,0,0,0,0,0,0
3,22,11751,58,3626,22,4206,17,10.0,6.2,0.49,0,0,0,0,0,0,0,0,0,0
4,14,19140,24,4211,10,6452,2,0.0,44.0,0.13,0,0,0,0,0,0,0,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature Notes

| Feature                  | Meaning                                                                                              | Missing-value handling                                | Available before prediction? |
| ------------------------ | ---------------------------------------------------------------------------------------------------- | ----------------------------------------------------- | ---------------------------- |
| `days_since_last_update` | Days since the page was last updated; higher values indicate older content.                          | Median imputation + missing flag                      | **Yes**                      |
| `impressions_90d`        | Search impressions over the last 90 days.                                                            | Median imputation + missing flag                      | **Yes**                      |
| `clicks_90d`             | Search clicks over the last 90 days.                                                                 | Median imputation + missing flag                      | **Yes**                      |
| `impressions_last_30d`   | Search impressions in the most recent 30 days.                                                       | Median imputation + missing flag                      | **Yes**                      |
| `clicks_last_30d`        | Search clicks in the most recent 30 days.                                                            | Median imputation + missing flag                      | **Yes**                      |
| `impressions_prev_30d`   | Search impressions in the previous 30-day period.                                                    | Median imputation + missing flag                      | **Yes**                      |
| `clicks_prev_30d`        | Search clicks in the previous 30-day period.                                                         | Median imputation + missing flag                      | **Yes**                      |
| `search_volume`          | Estimated search demand for the page's target keyword/topic.                                         | Median imputation + missing flag                      | **Yes**                      |
| `avg_position`           | Average search ranking position. `0` means no available position data.                               | Convert `0` to missing, then median imputation + flag | **Yes**                      |
| `ctr`                    | Click-through rate from search impressions to clicks. Stored as a percentage (e.g., `0.76` = 0.76%). | Median imputation + missing flag                      | **Yes**                      |

### Categorical handling

No categorical variables are required in this baseline feature vector. If `content_type` or `main_intent` is added later, they should be **one-hot encoded** rather than converted to arbitrary numeric values.

### Availability rule

All selected features represent information available **before the review/prediction point**, so they can be used for prioritization without using the leakage variables `trend_direction` or `trend_pct`.


In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [36]:
leakage_cols = ["trend_direction", "trend_pct"]

print("Leakage columns found:")
print([col for col in leakage_cols if col in df.columns])

# Make sure none are included in the feature vector
print("\nLeakage columns included in X:")
print(set(feature_cols) & set(leakage_cols))

Leakage columns found:
['trend_direction', 'trend_pct']

Leakage columns included in X:
set()


In [37]:
future_like = [
    col for col in feature_cols
    if any(word in col.lower() for word in ["future", "next", "forecast"])
]

excluded = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct"
]

used_excluded = [
    col for col in X.columns
    if col in excluded
]

print("Excluded fields accidentally used:", used_excluded)
print("Potential future-window features:", future_like)

Excluded fields accidentally used: []
Potential future-window features: []


### Leakage Hunt — Summary

I checked the feature set for information that could leak future or target information.
`trend_direction` and `trend_pct` were excluded because they represent trend information.
IDs and process fields such as `content_id`, `client_id`, `provider_used`, and `model_used` were excluded.
I also checked for future-looking fields such as `future`, `next`, or `forecast`.
The final feature vector contains only information available at the time of page prioritization.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* **`content_id`** — Identifier only; does not describe refresh opportunity.
* **`client_id`** — Identifies the client and could introduce client-specific bias.
* **`provider_used`** — Describes the data/provider source, not page opportunity.
* **`model_used`** — Describes the content-generation process, not current refresh need.
* **`search_volume`** — Describes keyword demand rather than whether the existing page needs refreshing.
* **`competition`** — Describes keyword competition rather than page condition.
* **`competition_level`** — Categorical version of competition; not a core refresh signal.
* **`cpc`** — Advertising-cost signal; not directly related to content freshness.
* **`content_type`** — Useful for segmentation, but excluded from the initial score to keep the ranking focused on page signals.
* **`main_intent`** — Describes search intent, but is not a direct measure of refresh opportunity.
* **`word_count`** — Content length alone does not show whether a page needs updating.
* **`char_count`** — Same as word count; length alone is insufficient evidence.
* **`ai_sessions_90d`** — AI traffic is outside the core signals for the initial refresh score.
* **`ai_traffic_pct`** — Same reason; not required for the initial ranking.
* **`scroll_events_90d`** — Useful engagement detail, but not necessary for the initial ranking.
* **`days_with_impressions`** — Supporting availability measure, not a core refresh signal.
* **`days_with_sessions`** — Supporting availability measure, not a core refresh signal.
* **`age_tier`** — Categorical representation of content age; duplicates other age information.
* **`age_tier_order`** — Derived from `age_tier`; redundant.
* **`freshness_tier`** — Categorical version of freshness already represented by `days_since_last_update`.
* **`word_count_tier`** — Derived from excluded word-count information.
* **`char_count_tier`** — Derived from excluded character-count information.
* **`impression_tier`** — Derived from impressions and therefore duplicates an included signal.
* **`position_tier`** — Derived from `avg_position` and therefore duplicates an included signal.



In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.